In [2]:
from pycromanager import Core
# select pattern kernel
core = Core()
devices = [core.get_loaded_devices().get(i) for i in range(core.get_loaded_devices().size())]
print(devices)
#z_device = core.get_focus_device()

# core.define_pixel_size_config("Objective-A", "DObjective", "Label", "Objective-A")
# core.set_pixel_size_um("Objective-A", 0.55)
# core.set_pixel_size_config("Objective-A")

stage = core.get_xy_stage_device()
print(f"Stage device: {stage}")
#print(dir(core))
#z = core.get_position(z_device)
x = core.get_x_position(stage)
y = core.get_y_position(stage)
print(f"Stage position: X={x}, Y={y}")
# core.set_origin(stage)
#core.set_origin_xy(stage)

# core.set_xy_position(stage, x, y+200)  # X=10 mm, Y=20 mm
# core.wait_for_device(stage)  # Wait for the stage to finish moving  

# x_new = core.get_x_position(stage)
# y_new = core.get_y_position(stage)
# print(f"Stage position: X={x_new}, Y={y_new}")

['COM9', 'COM8', 'XYStage', 'ZStage', 'Core']
Stage device: XYStage
Stage position: X=5247.400000000001, Y=-180.5


In [3]:
from pycromanager import Core

core = Core()
stage = core.get_xy_stage_device()
print(f"XY stage: {stage}")

x = core.get_x_position(stage)
y = core.get_y_position(stage)
print(f"Stage position: X={x}, Y={y}")

# Nudge +100 µm in Y
core.set_relative_xy_position(stage, 0, 10)
core.wait_for_device(stage)

x_new = core.get_x_position(stage)
y_new = core.get_y_position(stage)
print(f"After move: X={x_new}, Y={y_new}")

core.set_xy_position(x-200, y-200)
core.wait_for_device(stage)
x_new1 = core.get_x_position(stage)
y_new1 = core.get_y_position(stage)
print(f"After move: X={x_new1}, Y={y_new1}")



XY stage: XYStage
Stage position: X=0, Y=99.9
After move: X=0, Y=110
After move: X=-200, Y=-100.10000000000001


In [ ]:
core.set_adapter_origin_xy(stage, 0.0, 0.0)
x = core.get_x_position(stage)
y = core.get_y_position(stage)
print(f"Current XY position: X = {x:.2f} µm, Y = {y:.2f} µm")

core.set_xy_position(x-200, y-200) # setting absolute position crashes Micro-Manager
core.wait_for_device(stage)  # Wait for the stage to finish moving  
x = core.get_x_position(stage)
y = core.get_y_position(stage)
print(f"Current XY position: X = {x:.2f} µm, Y = {y:.2f} µm")

Current XY position: X = 0.00 µm, Y = 0.00 µm
Current XY position: X = -200.00 µm, Y = -200.00 µm


In [1]:
from pycromanager import Core
import time

def wait_ready(core, label, timeout=10):
    t0 = time.time()
    while core.device_busy(label):
        if time.time() - t0 > timeout:
            raise TimeoutError(f"{label} did not become ready in {timeout}s")
        time.sleep(0.05)

def move_rel(core, stage, dx_um, dy_um):
    core.set_relative_xy_position(stage, float(dx_um), float(dy_um))
    wait_ready(core, stage)

def goto_abs_emulated(core, stage, x_target_um, y_target_um):
    """Emulate absolute move using only relative commands."""
    x_now = core.get_x_position(stage)
    y_now = core.get_y_position(stage)
    dx = float(x_target_um) - float(x_now)
    dy = float(y_target_um) - float(y_now)
    # Two single-axis moves (often safer than a combined XY step)
    move_rel(core, stage, 0.0, dy)
    move_rel(core, stage, dx, 0.0)

# --- Test plan ---
core = Core()
stage = core.get_xy_stage_device()
if not stage:
    raise RuntimeError("No default XY stage configured.")
print(f"Stage: {stage}")

# Optional: slow down to be gentle during debugging (adapter allows this property)
try:
    core.set_property(stage, "MaxSpeed", 10000)  # µm/s; reduce if needed
    core.set_property(stage, "Acceleration", 200)
    print("Set conservative speed/acceleration.")
except Exception:
    pass  # Not all adapters expose these or allow writes

x0 = core.get_x_position(stage)
y0 = core.get_y_position(stage)
print(f"Start (µm): X={x0:.2f}  Y={y0:.2f}")

# 1) Prove relative control
print("Relative jog: +Y 20 µm, then -Y 20 µm")
move_rel(core, stage, 0.0, 20.0)
move_rel(core, stage, 0.0, -20.0)
x1 = core.get_x_position(stage); y1 = core.get_y_position(stage)
print(f"After relative test (µm): X={x1:.2f}  Y={y1:.2f}")

# 2) Emulated-absolute test: go to a nearby target and back (no negative-absolute commands issued)
x_t = x0 - 50.0
y_t = y0 + 50.0
print(f"Emulated absolute → ({x_t:.2f}, {y_t:.2f}) µm")
goto_abs_emulated(core, stage, x_t, y_t)
x2 = core.get_x_position(stage); y2 = core.get_y_position(stage)
print(f"Reached (µm): X={x2:.2f}  Y={y2:.2f}")

print("Return to start via emulated absolute")
goto_abs_emulated(core, stage, x0, y0)
x3 = core.get_x_position(stage); y3 = core.get_y_position(stage)
print(f"Returned (µm): X={x3:.2f}  Y={y3:.2f}")


Stage: XYStage
Set conservative speed/acceleration.
Start (µm): X=-8700.20  Y=105.50
Relative jog: +Y 20 µm, then -Y 20 µm
After relative test (µm): X=-8700.20  Y=105.50
Emulated absolute → (-8750.20, 155.50) µm
Reached (µm): X=-8750.20  Y=155.50
Return to start via emulated absolute
Returned (µm): X=-8700.20  Y=105.50
